<h3 style="color: #3498db;"> 2.1.1 入门 </h3>

In [ ]:
#检测pytorch和cuda版本号
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [ ]:
y1=torch.zeros((2,3,4))#创建一个全零张量
y2=torch.ones((2,3,4))#创建一个全一张量

print(y1.shape)
print(y1.numel())


torch.Size([2, 3, 4])
24


In [ ]:
torch.randn(3,4)#创建一个服从标准正态分布的张量
torch.tensor([[2,1,4,3],[1,2,3,4],[4,3,2,1]])#赋予确定的元素值

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

<h3 style="color: #3498db;"> 2.1.2 运算符 </h3>

* **逐元素运算**：`+` `-` `*` `/` `**` 均为 Shape 严格对齐的逐元素操作，底层由 CUDA 万核并发执行。
* **拼接 (Concatenate)**：`torch.cat((X, Y), dim=d)` 沿第 `d` 维物理拼接，该维度长度相加，其余维度必须严格一致。
* **逻辑判断**：`X == Y` 逐元素比较，返回同 Shape 的布尔张量（`True/False`）。
* **全局求和**：`X.sum()` 将所有元素坍缩为单个标量张量，Shape 从 $(m, n)$ 降维至 $()$。

In [ ]:
x=torch.tensor([1.0,2,4,8])
y=torch.tensor([2,2,2,2])
x+y,x-y,x*y,x/y,x**y

torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

In [ ]:
X=torch.arange(12,dtype=torch.float32).reshape((3,4))#定义X变量
Y=torch.tensor([[2.0,1,4,3],[1,2,3,4],[4,3,2,1]])#定义Y变量
torch.cat((X,Y),dim=0),torch.cat((X,Y),dim=1)#dim=0-行连结，dim=1-列连结

X==Y#判断是否相等，相等为true，否则为false

X.sum()#求和，变成一个单元素张量

tensor(66.)

<h3 style="color: #3498db;"> 2.1.3 广播机制 (Broadcasting) </h3>

**触发条件**：两个 Shape 不同的张量执行逐元素运算时，底层自动沿**长度为 1 的维度**进行虚拟复制对齐。

* **物理动作**：不实际分配新内存，仅在计算时"假装"沿缺失维度复制扩展，开销为零。
* **Strang 映射**：本质是向量空间中的外积式升维——列向量沿行方向广播 = 将 $\mathbb{R}^{m \times 1}$ 虚拟扩展为 $\mathbb{R}^{m \times n}$。
* **工程陷阱**：当两个张量的维度既不相等也不为 1 时，广播失败，直接抛出 Shape 不匹配的 RuntimeError。

In [ ]:
a=torch.arange(3).reshape((3,1))
b=torch.arange(2).reshape((1,2))
print(a,'\n',b)
a+b# a->3*2 ,b->3*2

tensor([[0],
        [1],
        [2]]) 
 tensor([[0, 1]])


tensor([[0, 1],
        [1, 2],
        [2, 3]])

<h3 style="color: #3498db;"> 2.1.4 索引和切片 </h3>

* **单点索引** `x[i, j]`：触发**维度坍缩**，被索引的维度直接消失，返回降维后的子张量。
* **切片索引** `x[i:j]`：**强制保留维度骨架**，即使只切出一个元素，Shape 中该维度仍以长度 1 存在。
* **左闭右开铁律**：`x[a:b]` 物理上提取索引 $[a, b)$，即包含 $a$ 但不包含 $b$。
* **赋值广播**：`x[0:2, :] = 12` 底层对右侧标量触发广播，批量覆写指定区域的所有元素。

In [ ]:
X[-1],X[1:3]#[-1]选择最后一行元素,[-3]选择第二行和第三行元素
X[1,2]=9#单元素赋值
X

X[0:2,:]=12
X

tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

<h3 style="color: #3498db;"> 2.1.5 张量的内存管理 (In-place Operations) </h3>

**核心痛点**：非原地操作频繁开辟新显存，极易引发碎片化与 OOM 崩溃。

| 操作 | 底层机制 | 指针变化 | 显存开销 |
|------|----------|----------|----------|
| `Y = Y + X` | 新开辟完整内存，指针改向 | ✅ 改变 | 最差 |
| `Y[:] = Y + X` | 临时内存计算后拷贝回原地址 | ❌ 不变 | 峰值有开销 |
| `Y += X` / `Y.add_(X)` | 直接在原内存格子内覆写，零临时分配 | ❌ 不变 | **零额外开销，工程最优解** |

In [ ]:
before=id(Y)
Y=Y+X#非原地操作，重新分配内存
id(Y)==before#返回false

Y+=X#原地操作，内存地址不变
Y[:]=Y+X#同上
id(Y)==before

True

In [ ]:
#示例：
Z=torch.zeros_like(Y)
print('id(Z):',id(Z))
Z[:]=X+Y
print('id(Z):',id(Z))

id(Z): 2164800333328
id(Z): 2164800333328


<h3 style="color: #3498db;"> 2.1.6 转换为其他 Python 对象 </h3>

### 核心结论：三条 API 边界

| 场景 | API | 输出类型 | 底层行为 |
|------|-----|----------|----------|
| Tensor → NumPy | `.numpy()` | `np.ndarray` | **共享底层内存**，原地修改互相影响 |
| NumPy → Tensor | `torch.tensor(A)` | `torch.Tensor` | 同上，共享内存 |
| 单元素张量 → 纯数字 | `.item()` | Python `float/int` | 斩断计算图，安全提取（防 OOM）|
| 多元素张量 → 纯列表 | `.tolist()` | Python `list` | 批量剥离为原生嵌套列表 |

In [ ]:
#type()验证格式转换
A=X.numpy()
B=torch.tensor(A)
type(A),type(B)

(numpy.ndarray, torch.Tensor)

In [ ]:
#转换大小为1的张量
a=torch.tensor([3.5])
a,a.item(),float(a),int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

In [ ]:
x=torch.tensor([[1,2],[3,4]])
x[1,1].item()#提取单元素
x.tolist()#提取列表

[[1, 2], [3, 4]]

<h1 align="center"> 🛠️ 2.2 数据预处理 </h1>
<hr>

<h3 style="color: #3498db;"> 2.2.1 读取数据集 </h3>

### 1. 文件系统 I/O (`os` 模块)
* `os.path.join`：跨平台路径拼接，自动适配 `\` 或 `/`。`..` = 返回上一级父目录。
* `os.makedirs`：附带 `exist_ok=True`，目录已存在时静默跳过，不抛异常。
* `with open(...) as f`：上下文管理器，无论成功或报错，自动关闭文件句柄，杜绝资源泄漏。

### 2. Pandas 读取 (`pd.read_csv`)
* **工程定位**：Pandas 专司数据清洗脏活，清洗完毕后必须转化为 PyTorch Tensor 才能送入模型。
* **读取机制**：严格服从变量中的物理路径，自动解析首行为表头列名，自动将空缺内容映射为 `NaN`。

In [ ]:
#写入数据集
import os

os.makedirs(os.path.join('..','data'),exist_ok=True)
data_file=os.path.join('..','data','house_tiny.csv')
with open(data_file,'w') as f:
    f.write('NumRooms,Alley,Price\n')#列名
    f.write('NA,Pave,127500\n')#每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')


In [ ]:
#!pip install pandas #安装pandas
import pandas as pd
#读取数据集
data=pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


<h3 style="color: #3498db;"> 2.2.2 处理缺失值 (Data Preprocessing) </h3>
神经网络底层拒收缺失值（`NaN`）与文本。本节核心是将残缺业务表格淬炼为纯净张量：

### 1. 架构解耦 ($X$ 与 $Y$)
* **物理动作**：通过 `.iloc` 坐标切片，将数据强行切断为“输入特征 $X$” (`inputs`) 与“预测目标 $Y$” (`outputs`)。
* **工程铁律**：后续所有预处理只能针对 $X$。绝不允许 $Y$ 混入，严防数据泄露（Data Leakage）。

### 2. 均值插补 (数值特征)
* **物理动作**：调用 `.fillna()`，用数值列的**均值**覆盖 `NaN` 空洞，维持分布的“几何质心”。
* **断层规避**：Pandas 2.0+ 必须显式下达 `numeric_only=True` 指令，强制跳过非数值列。

### 3. 独热编码 (文本特征)
* **物理动作**：调用 `pd.get_dummies()`，将 $N$ 种文本状态撕裂为 $N$ 维正交基（独立的 0/1 特征列）。
* **情报固化**：开启 `dummy_na=True`，将 `NaN` 视为独立状态保留。在工业界，数据的缺失行为本身即是强特征。

In [ ]:
#神经网络架构Y=WX
# inputs 接管了第 0 列和第 1 列（房间数和巷子类型)--X
# outputs 独占了第 2 列（价格）--Y
inputs,outputs=data.iloc[:,0:2],data.iloc[:,2]
print(inputs.dtypes)
inputs=inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

NumRooms    float64
Alley        object
dtype: object
   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [ ]:
#独热编码（One-Hot Encoding）
inputs=pd.get_dummies(inputs,dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


<h3 style="color: #3498db;"> 2.2.3 转换为张量格式 (核心数据流) </h3>
数据在送入 GPU 前，必须经历三次**严格单向**的物理蜕变：

1. **Pandas DataFrame**（初始 `inputs`）：**业务清洗台**。保留列名，包容 `NaN` 和文本，专职负责特征解耦与清洗。
2. **NumPy Array**（`.to_numpy()` 算子）：**剥离业务属性**。强行扒掉列名与行号，退化为绝对纯粹的浮点数连续内存块。
3. **PyTorch Tensor**（`torch.tensor()`）：**GPU 算力弹药**。赋予底层求导（Gradient）器官，具备送入显卡执行万核并发矩阵乘法（$Y=WX$）的物理资格。

> **工程铁律**：Pandas 只干清洗脏活，Tensor 专责核心算力。物理边界森严，绝不混用。

In [ ]:
import torch
#注意：需先进行上文的独热编码，把文本撕裂成 True/False 的数字维度
X=torch.tensor(inputs.to_numpy(dtype=float))
y=torch.tensor(outputs.to_numpy(dtype=float))
X,y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

<h1 align="center"> 📐 2.3 线性代数 (Linear Algebra) </h1>
<hr>

<h3 style="color: #3498db;"> 2.3.1 标量——由单个元素表示 </h3>

In [2]:
#单元素标量运算
import torch
x=torch.tensor(3.0)
y=torch.tensor(2.0)
x+y,x*y,x/y,x**y

(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

<h3 style="color: #3498db;"> 2.3.2 向量——一维张量 </h3>

In [ ]:
#创建一维张量
x=torch.arange(4)
x

tensor([0, 1, 2, 3])

In [ ]:
x[3]

tensor(3)

In [ ]:
len(x)#访问张量长度
x.shape#张量形状

torch.Size([4])

In [ ]:
A=torch.arange(20).reshape(5,4)
A

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [ ]:
A.T#转置

tensor([[ 0,  4,  8, 12, 16],
        [ 1,  5,  9, 13, 17],
        [ 2,  6, 10, 14, 18],
        [ 3,  7, 11, 15, 19]])

In [ ]:
B=torch.tensor([[1,2,3],[2,0,4],[3,4,5]])
B

tensor([[1, 2, 3],
        [2, 0, 4],
        [3, 4, 5]])

In [ ]:
B==B.T#判断对称矩阵

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])

<h3 style="color: #3498db;"> 2.3.4 张量 (Tensors) </h3>

**核心定义**：标量（0阶）、向量（1阶）、矩阵（2阶）向 $n$ 维坐标轴的物理泛化。

* **物理结构**：一段连续的底层显存，依靠 `Shape` 元组来决定如何“折叠”读取。例如 `reshape(2, 3, 4)` 意味着将 24 个元素在逻辑上切分为 2 大块，每块 3 行 4 列。
* **工程映射 (视觉领域)**：
  - 3 阶张量 `(C, H, W)`：单张图像（通道数, 高度, 宽度）。
  - 4 阶张量 `(B, C, H, W)`：深度学习标准数据流骨架，额外增加了批次大小（Batch Size）。
* **计算铁律**：不论张量达到多少阶，在 GPU 底层运算时，都会被完全拍平（Flatten）为一维连续数组进行并发计算。

In [ ]:
X=torch.arange(24).reshape(2,3,4)
X

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

<h3 style="color: #3498db;"> 2.3.5 张量算法的基本性质 </h3>

* **Hadamard 积 (`*`)**：严格的逐元素相乘。其核心数学特性是计算前后张量的**多维空间骨架（Shape）绝对守恒**，这与改变维度的矩阵点积运算（`@`）有本质物理区别。
* **标量运算的本质**：张量与标量（如 `a=2`）运算时，标量在底层被打包为 `()` 的 0 维张量，通过**广播机制**自动虚拟膨胀至相同维度，实现零额外开销的全局逐元素缩放。
* **显存硬隔离 (`.clone()`)**：强制在底层显存中开辟一块全新的物理内存进行深拷贝，彻底斩断与原张量的内存指针绑定，杜绝原地操作带来的连带污染。
* **广播机制物理约束**：无论维度多复杂，能否触发广播底层只看一条铁律——**从右向左逐维比对**，在每一层空间对齐上，两者长度必须“相等”或“其中之一严格等于 1”，否则立刻报错阻断计算。

In [4]:
A=torch.arange(20,dtype=torch.float32).reshape(5,4)
B=A.clone()#通过分配新内存，将A的一个副本分配给B
A,A+B

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [12., 13., 14., 15.],
         [16., 17., 18., 19.]]),
 tensor([[ 0.,  2.,  4.,  6.],
         [ 8., 10., 12., 14.],
         [16., 18., 20., 22.],
         [24., 26., 28., 30.],
         [32., 34., 36., 38.]]))

In [ ]:
A*B#Hadamard积,逐元素相乘

tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.],
        [144., 169., 196., 225.],
        [256., 289., 324., 361.]])

In [ ]:
a=2
X=torch.arange(24).reshape(2,3,4)
a+X,(a*X).shape

(tensor([[[ 2,  3,  4,  5],
          [ 6,  7,  8,  9],
          [10, 11, 12, 13]],
 
         [[14, 15, 16, 17],
          [18, 19, 20, 21],
          [22, 23, 24, 25]]]),
 torch.Size([2, 3, 4]))

<h3 style="color: #3498db;"> 2.3.6 降维 </h3>

* **降维求和（Reduction）**：
  * 默认情况下，`.sum()` 沿所有轴将张量降维为一个标量。
  * 指定 `axis=0`（或 `dim=0`）沿行方向坍缩（纵向求和），消除第 0 维，得到 `(列数,)` 形状的向量。
  * 指定 `axis=1`（或 `dim=1`）沿列方向坍缩（横向求和），消除第 1 维，得到 `(行数,)` 形状的向量。
* **工程优化与防坑**：
  * **求和效率**：全局求和 `A.sum()` 性能优于 `A.sum(axis=[0, 1])`。前者直接调用全局规约算子 `sum_all`，无需步长寻址，开销极低；后者走通用的 `sum_dim` 分维度计算，带有 Python 参数解析与 C++ 步长元数据计算开销。
  * **均值计算**：计算平均值时应直接调用 `A.mean()`，严禁使用 `A.sum() / A.numel()`。原生 `mean` 仅需一次算子调度且底层会自动开启防溢出保护（如使用高精度累加器），而拼装法会带来三次调度开销，且在大张量下第一步 `sum()` 极易发生数值溢出（Overflow）爆成 `inf`。

In [5]:
#单维求和
x=torch.arange(4,dtype=torch.float32)
x,x.sum()

(tensor([0., 1., 2., 3.]), tensor(6.))

In [ ]:
#多维求和，使用sum()函数默认沿所有轴降维求和--变成单一标量
A.shape,A.sum()

(torch.Size([5, 4]), tensor(190.))

In [ ]:
A_sum_axis0=A.sum(axis=0)#axis锁定轴0,加入keepdims=True保留一维
A_sum_axis0,A_sum_axis0.shape

(tensor([40., 45., 50., 55.]), torch.Size([4]))

In [ ]:
A_sum_axis1=A.sum(axis=1)#axis锁定轴1
A_sum_axis1,A_sum_axis1.shape

(tensor([ 6., 22., 38., 54., 70.]), torch.Size([5]))

In [ ]:
A.sum(axis=[0,1])#结果与A.sum()相同，但sum求和优于axis

tensor(190.)

In [ ]:
#求平均值
A.mean(),A.sum()/A.numel()#A.mean() 的性能和安全性都稳压 A.sum()/A.numel()

(tensor(9.5000), tensor(9.5000))

In [ ]:
#沿指定轴求平均值
A.mean(axis=0),A.sum(axis=0)/A.shape[0]#A.mean(axis=0) 在性能和安全性上依然完胜。

(tensor([ 8.,  9., 10., 11.]), tensor([ 8.,  9., 10., 11.]))

In [8]:
#非降维求和
sum_A=A.sum(axis=1,keepdims=True)
sum_A

tensor([[ 6.],
        [22.],
        [38.],
        [54.],
        [70.]])

In [9]:
A/sum_A

tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])

In [13]:
A.cumsum(axis=0)#不会沿任何轴降低输入张量的维度

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])

<h3 style="color: #3498db;"> 2.3.7 点积 </h3>

* **基本定义与 API 约束**：
  * **数学定义**：点积（Dot Product）为相同位置按元素相乘的和，物理表达式为 $\mathbf{x}^\top \mathbf{y} = \sum_{i=1}^d x_i y_i$。
  * **API 约束**：`torch.dot(x, y)` 仅支持**两个一维向量**求点积。二维矩阵乘法必须使用 `torch.mm`、`torch.matmul` 或运算符 `@`。
* **性能差异与底层实现**：
  * **显存分配**：`torch.dot` 空间复杂度为 $O(1)$，通过寄存器在片上直接在线累加；而 `torch.sum(x * y)` 空间复杂度为 $O(N)$，需强行分配一块等大的临时内存存放中间乘积结果，浪费显存带宽。
  * **算子融合**：`torch.dot` 底层调用硬件高度优化的 BLAS 库（如 `cublasSdot`），是一体化的单内核（Kernel）操作；而拼装法需要分别调度乘法和求和两个 Kernel，多出一倍的通信调度延迟。
* **深度学习中的三大物理意义**：
  * **加权和（Weighted Sum）**：单个神经元整合不同特征（$\mathbf{x}^\top \mathbf{w}$）的基础计算形式。
  * **加权平均（Weighted Average）**：当权重非负且和为 1 时，点积退化为加权均值（如 Transformer 注意力分布对 Value 向量的加权整合）。
  * **夹角余弦（Cosine Similarity）**：向量规范化为单位长度后，点积公式 $\mathbf{a}^\top \mathbf{b} = \|\mathbf{a}\| \|\mathbf{b}\| \cos\theta$ 简化为 $\mathbf{u}^\top \mathbf{v} = \cos\theta$，即大模型 Embedding 语义相似度匹配的物理数学基石。

In [ ]:
y=torch.ones(4,dtype=torch.float32)
x,y,torch.dot(x,y)#dot用于求一维向量的点积（内积）

(tensor([0., 1., 2., 3.]), tensor([1., 1., 1., 1.]), tensor(6.))

In [14]:
torch.sum(x*y)#在性能和资源开销上，torch.dot(x, y) 完胜 torch.sum(x * y)

tensor(6.)

<h3 style="color: #3498db;"> 2.3.8 矩阵-向量积 </h3>

* **计算规则**：矩阵 $\mathbf{A} \in \mathbb{R}^{m \times n}$ 与向量 $\mathbf{x} \in \mathbb{R}^{n}$ 的乘积 $\mathbf{Ax}$，本质是用矩阵的每一行分别与向量 $\mathbf{x}$ 求一次点积，输出一个长度为 $m$ 的新向量。
* **Shape 变换**：$(m, n) \times (n,) \rightarrow (m,)$。矩阵的列数必须严格等于向量的长度，否则底层维度校验直接崩溃。
* **API**：`torch.mv(A, x)`，专用于二维矩阵与一维向量的乘积。
* **空间转换视角**：矩阵 $\mathbf{A}$ 充当线性映射（传送门），将 $n$ 维空间中的向量 $\mathbf{x}$ 投射到 $m$ 维空间中。输出结果被物理约束在矩阵 $\mathbf{A}$ 的列空间 $\mathcal{C}(A)$ 之内。

In [ ]:
A.shape,x.shape,torch.mv(A,x)#求向量积

(torch.Size([5, 4]), torch.Size([4]), tensor([ 14.,  38.,  62.,  86., 110.]))

<h3 style="color: #3498db;"> 2.3.9 矩阵-矩阵乘法 </h3>

* **空间变换规则**：$(n, k) \times (k, m) \rightarrow (n, m)$。内侧维度 $k$ 必须严丝合缝地物理对齐，本质是执行了 $m$ 次矩阵-向量积并横向拼接。
* **双重 API 工程生态**：
  * **`torch.mm(A, B)`（严格模式）**：极其死板，强制校验输入必须是标准的二维张量，常用于刻意拦截高维数据防止逻辑出错。
  * **`A @ B`（工程首选）**：底层调用 `torch.matmul`。它是具备智能嗅探的现代算子，能自动向下兼容点积与矩阵向量积，更能**向上支持带 Batch（批次）的高维批量矩阵乘法（BMM）**。在真实的大模型实操中，`@` 是绝对的标准写法。

In [ ]:
B=torch.ones(4,3)
torch.mm(A,B)#只接受二维张量

tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])

<h3 style="color: #3498db;"> 2.3.10 范数 </h3>

* **核心定义**：范数是"向量的长度"，把一个多元张量压缩成单一标量来衡量其"大小"。

* **两大核心范数对比**：

| 范数 | 公式 | API | 特性 |
|------|------|-----|------|
| $L_2$ 范数 | $\|\mathbf{x}\|_2 = \sqrt{\sum x_i^2}$ | `torch.norm(x)` | 对异常值极度敏感（平方放大） |
| $L_1$ 范数 | $\|\mathbf{x}\|_1 = \sum \|x_i\|$ | `torch.abs(x).sum()` | 对异常值鲁棒，倾向于将权重逼为 0（稀疏化） |

* **Frobenius 范数**：矩阵专属的"$L_2$ 范数"，将所有元素平方求和再开根号：$\|\mathbf{X}\|_F = \sqrt{\sum_{i,j} x_{ij}^2}$，调用方式与向量完全一致：`torch.norm(torch.ones((4, 9)))`。

* **范数在深度学习中的工程意义（正则化防过拟合）**：
  * **$L_2$ 正则化（权重衰减）**：对大权重惩罚更重，促使所有权重**均匀收缩变小**。底层等价于对 $A^TA$ 加上 $\lambda I$，强制恢复矩阵正定性，防止解空间崩塌。
  * **$L_1$ 正则化**：持续压榨小权重直到归零，自动完成**特征选择（稀疏化）**。
  * **深度学习的目标函数**本身通常就是一个范数：最小化预测值与真实值之间的 $L_2$ 距离。

In [17]:
u=torch.tensor([3.0,-4.0])
torch.norm(u)

tensor(5.)

In [18]:
torch.abs(u).sum()

tensor(7.)